In [ ]:
import pandas as pd
df=pd.read_csv("C:\나이스\summary_experience_dataset.csv")
qa_df = df[['answer_text', 'summary_text']].copy()
print(qa_df.head())

In [ ]:
import pandas as pd
from datasets import Dataset, DatasetDict

# --- 이전 단계에서 qa_df를 생성하는 코드 ---
df=pd.read_csv("C:\나이스\summary_experience_dataset.csv")
qa_df = df[['answer_text', 'summary_text']].copy()
print(qa_df.head())
# ----------------------------------------

# 1. Pandas DataFrame을 Hugging Face Dataset 객체로 변환
# Dataset.from_pandas()를 사용하여 DataFrame을 Dataset으로 변환합니다.
# 이 때, 데이터프레임의 인덱스는 무시됩니다.
full_dataset = Dataset.from_pandas(qa_df)

# 변환된 Dataset 확인
print("Full Dataset:")
print(full_dataset)
print(full_dataset[0]) # 첫 번째 샘플 확인

# 2. Dataset을 훈련(train) 및 검증(validation) 세트로 분할
# train_test_split 메서드를 사용하여 데이터셋을 분할할 수 있습니다.
# test_size는 검증 세트의 비율을 나타냅니다. (예: 0.1이면 10%가 검증 세트)
# seed를 설정하면 매번 동일한 분할 결과를 얻을 수 있습니다.
train_test_split_dataset = full_dataset.train_test_split(test_size=0.1, seed=42)

# 3. 분할된 데이터셋을 DatasetDict 형태로 저장 (선택 사항이지만 일반적)
# Trainer API를 사용하려면 DatasetDict 형태가 편리합니다.
raw_datasets = DatasetDict({
    'train': train_test_split_dataset['train'],
    'validation': train_test_split_dataset['test'] # train_test_split은 'test' 키를 사용합니다.
})

# 최종 raw_datasets 구조 확인
print("\nFinal raw_datasets (DatasetDict):")
print(raw_datasets)
print("\nTrain dataset example:")
print(raw_datasets['train'][0])
print("\nValidation dataset example:")
print(raw_datasets['validation'][0])


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, TrainingArguments, Trainer, DataCollatorForSeq2Seq
import torch
import numpy as np
import evaluate
import nltk # ROUGE 스코어 계산을 위해 필요

# NLTK punkt 토크나이저 다운로드 (최초 1회 실행)
try:
    nltk.data.find('tokenizers/punkt')
except nltk.downloader.DownloadError:
    nltk.download('punkt')
    print("NLTK 'punkt' tokenizer downloaded.")

# 모델 체크포인트 설정
model_checkpoint = "eenzeenee/t5-base-korean-summarization"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# 최대 입력 및 타겟 길이 설정
# T5 모델은 일반적으로 512 토큰까지 처리할 수 있습니다.
# 요약문의 길이는 데이터셋 특성에 따라 조절하세요.

def preprocess_function(examples):
    # 'answer_text'를 입력 텍스트로 사용하고 "summarize: " 접두사 추가
    # T5 모델은 특정 작업을 위해 이러한 접두사를 사용합니다.
    inputs = ["summarize: " + doc for doc in examples["answer_text"]]
    model_inputs = tokenizer(inputs, truncation=True)

    # 'summary_text'를 요약문 (레이블)으로 사용하고 토큰화
    labels = tokenizer(text_target=examples["summary_text"], truncation=True)

    # 모델 학습을 위해 토큰화된 레이블을 'labels' 키에 할당
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# 모든 데이터셋 분할에 대해 전처리 함수 적용
print("\n토큰화 시작...")
tokenized_datasets = raw_datasets.map(preprocess_function, batched=True)
print("토큰화 완료.")

print("\n토큰화된 데이터셋 구조:")
print(tokenized_datasets)
print("\n토큰화된 훈련 데이터셋 첫 번째 샘플:")
print(tokenized_datasets['train'][0])


토큰화 시작...


Map: 100%|██████████| 917/917 [00:00<00:00, 5428.83 examples/s]

토큰화 완료.

토큰화된 데이터셋 구조:
DatasetDict({
    train: Dataset({
        features: ['answer_text', 'summary_text', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 8250
    })
    validation: Dataset({
        features: ['answer_text', 'summary_text', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 917
    })
})

토큰화된 훈련 데이터셋 첫 번째 샘플:
{'answer_text': '저 같은 경우에는 팀장보다는 팀원에 좀 더 잘 맞는 사람입니다. 그 이유는 저는 제 자신의 능력에 대해서 잘 알고 있는데요. 물론 저 역시도 어렸을 때는 팀장을 꿈꾸는 사람이었습니다. 제가 여기서 말하는 팀장이라는 건 정말 직책적인 그런 의미가 아니고 리더로서의 역할을 의미합니다. 그런 의미에서 볼 때 제가 자기 주도 능력이 없는 그런 리더십이 없는 사람이 아니고 리더십이 있지만 나는 내가 팔로워로서 활동을 할 때가 더 큰 성과를 낼 수 있다 라는 점에 대해서 너무 잘 알고 있었기 때문에 방금과 같이 대답을 하였습니다.', 'summary_text': '저는 리더십이 없지만 팔로워로서 활동을 할 때가 더 큰 성과를 낼 수 있다는 점을 알고 있어서 팀원에 좀 더 잘 맞는 사람이라고 생각합니다.', 'input_ids': [7675, 78, 20359, 74, 26159, 27, 222, 425, 222, 741, 222, 774, 690, 222, 4490, 2067, 222, 11200, 279, 222, 759, 222, 451, 222, 592, 222, 2055, 222, 660, 535, 15, 222, 337, 222, 1195, 274, 222, 425, 274, 222, 354, 222, 992, 